In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from claude_agent_sdk import (
    ClaudeSDKClient,
    SystemMessage,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ClaudeAgentOptions,
    UserMessage,
    ToolUseBlock,
    ToolResultBlock,
    AgentDefinition,
)

researcher = AgentDefinition(
    description="Researches a subtopic using web search and fetches sources.",
    prompt=(
        "You are a research assistant. Given a subtopic, use WebSearch to find "
        "2-3 relevant, recent sources. Use WebFetch to read each source. "
        "Write a markdown notes file with your findings to the path you are given. "
        "Include: key facts, direct quotes where relevant, and source URLs. "
        "Limit yourself to 3 WebSearch calls maximum."
    ),
    permissionMode="acceptEdits",
    tools=["WebFetch", "WebSearch", "Write"],
    maxTurns=15,
)


options = ClaudeAgentOptions(
    allowed_tools=["Write", "Agent"],
    agents={
        "researcher": researcher,
    },
)

orchestrator_prompt = f"""
Research the topic: "rare earths"

Steps:
1. Break the topic into 2 subtopics.
2. Spawn one `researcher` subagent per subtopic. Pass each a task like:
   "Research [subtopic]. Write your findings to [subtopic_slug].md"
3. After both finish, synthesize their findings into summary.md.
   The summary should have: an intro paragraph, key findings as bullets, and sources.
"""


async with ClaudeSDKClient(options=options) as client:

    await client.query(prompt=orchestrator_prompt)

    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"AssistantMessage TextBlock = {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"AssistantMessage ToolUseBlock = {block.name} {block.input}")
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"UserMessage ToolResultBlock = {block.content}")

AssistantMessage TextBlock = I'll break "rare earths" into two subtopics and spawn researchers for each in parallel.

**Subtopic 1:** Rare Earth Elements — geology, mining, and global supply chain
**Subtopic 2:** Rare Earths — strategic importance, geopolitics, and the race for independence
AssistantMessage ToolUseBlock = Agent {'subagent_type': 'researcher', 'description': 'Research rare earths geology/mining', 'prompt': "Research the following subtopic thoroughly and write your findings to a markdown file.\n\n**Subtopic:** Rare Earth Elements — Geology, Mining, and Global Supply Chain\n\nCover these areas:\n- What are rare earth elements (the 17 elements, their categories: light vs heavy)\n- Where rare earth deposits are found globally (major reserves by country)\n- How rare earths are mined and processed (open-pit, in-situ leaching, the separation/refining challenge)\n- Environmental impacts of rare earth mining (toxic waste, radioactive tailings, water contamination)\n- Global supp